## Loading the source extracts

The cell below reads every CSV file inside the ZIP archive into a dictionary of DataFrames, using each filename (without `.csv`) as the table key. It reports the number of tables and the dimensions of each extract. If a file is missing or the archive path is incorrect, loading will fail and should be resolved before continuing.

In [4]:
import zipfile
from pathlib import Path
import pandas as pd
import numpy as np

# Path to uploaded stadioequities.zip
zip_path = Path("stadioequities.zip")

# Read all nine CSV files directly from the stadioequities.zip
with zipfile.ZipFile(zip_path, "r") as z:
    csv_files = sorted(
        name for name in z.namelist()
        if name.lower().endswith(".csv")
    )

    tables = {
        Path(name).stem: pd.read_csv(
            z.open(name),
            dtype={"client_id": "string"}
        )
        for name in csv_files
    }

print("Number of CSV tables:", len(tables))

for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")


Number of CSV tables: 9
1_client_account: 2,110 rows, 9 columns
2_app_web_behaviour: 27,645 rows, 10 columns
3_funding_account_activity: 4,863 rows, 7 columns
4_trading_activity: 4,798 rows, 9 columns
5_product_subscription: 960 rows, 6 columns
6_support_feedback: 1,299 rows, 7 columns
7_marketing_activity: 10,372 rows, 7 columns
8_compliance_monitoring: 130 rows, 5 columns
9_modelling_outcomes: 2,104 rows, 6 columns


In [5]:
# Clean the client_id values and remove exact duplicate rows
for name, df in tables.items():
    if "client_id" not in df.columns:
        raise ValueError(f"{name} does not contain client_id.")

    df["client_id"] = df["client_id"].str.strip()

    # Treat blank IDs as missing
    df["client_id"] = df["client_id"].replace("", pd.NA)

    # Remove records that are identical across every column
    tables[name] = df.drop_duplicates().copy()

# Check missing primary keys
for name, df in tables.items():
    print(f"{name}: {df['client_id'].isna().sum()} missing client IDs")

# Remove records without a join key
for name, df in tables.items():
    tables[name] = df.dropna(subset=["client_id"]).copy()

# The client account table should contain one row per client.
# The six duplicate client IDs in the supplied extract are exact duplicates.
client_account = tables["1_client_account"].drop_duplicates(
    subset=["client_id"],
    keep="first"
).copy()

tables["1_client_account"] = client_account

print("\nUnique clients in client account table:",
      client_account["client_id"].nunique())


1_client_account: 0 missing client IDs
2_app_web_behaviour: 0 missing client IDs
3_funding_account_activity: 0 missing client IDs
4_trading_activity: 0 missing client IDs
5_product_subscription: 0 missing client IDs
6_support_feedback: 0 missing client IDs
7_marketing_activity: 0 missing client IDs
8_compliance_monitoring: 0 missing client IDs
9_modelling_outcomes: 0 missing client IDs

Unique clients in client account table: 2104


In [6]:
def aggregate_by_client(df, prefix, exclude_columns=None):
    """
    Convert a transaction/event-level table into one row per client.
    All generated feature names receive a table-specific prefix.
    """
    exclude_columns = set(exclude_columns or [])
    feature_columns = [
        c for c in df.columns
        if c not in {"client_id"} | exclude_columns
    ]

    work = df[["client_id"] + feature_columns].copy()

    # Identify and convert date/time columns
    datetime_cols = [
        c for c in feature_columns
        if "timestamp" in c.lower() or "date" in c.lower()
    ]

    for col in datetime_cols:
        work[col] = pd.to_datetime(work[col], errors="coerce")

    aggregations = {}

    for col in feature_columns:
        if col in datetime_cols:
            aggregations[col] = ["min", "max"]

        elif pd.api.types.is_numeric_dtype(work[col]) or \
                pd.api.types.is_bool_dtype(work[col]):
            # Acquisition cost is client-level and repeated across
            # campaign rows, so do not sum it.
            if col == "acquisition_cost":
                aggregations[col] = ["mean", "max"]
            else:
                aggregations[col] = ["count", "sum", "mean", "max"]

        else:
            # For categories and text, retain distinct counts and
            # the most frequent value rather than every event.
            aggregations[col] = [
                ("nunique", lambda s: s.nunique(dropna=True)),
                ("mode", lambda s: (
                    s.mode(dropna=True).iloc[0]
                    if not s.mode(dropna=True).empty
                    else pd.NA
                ))
            ]

    # Build a one-row-per-client summary
    grouped = work.groupby("client_id").agg(aggregations)

    # Flatten the generated multi-level column names
    grouped.columns = [
        f"{prefix}_{col}_{stat}"
        for col, stat in grouped.columns
    ]

    return grouped.reset_index()


## Building the consolidated client table

The client-account table provides the base population. Each operational table is aggregated and left-joined on `client_id`; the outcomes table is then joined after checking for duplicate outcome keys. One-to-one validation is used to catch unexpected key relationships. Inspect the final shape and sample rows before proceeding.

In [7]:
# Start with the client profile table
merged_df = tables["1_client_account"].copy()

# Aggregate the seven operational tables
operational_tables = [
    ("2_app_web_behaviour", "app"),
    ("3_funding_account_activity", "funding"),
    ("4_trading_activity", "trading"),
    ("5_product_subscription", "subscription"),
    ("6_support_feedback", "support"),
    ("7_marketing_activity", "marketing"),
    ("8_compliance_monitoring", "compliance"),
]

for table_name, prefix in operational_tables:
    df = tables[table_name]

    # first_deposit_timestamp is also present in the outcomes table.
    # Exclude it here to avoid duplicating the activation-related field.
    exclusions = (
        ["first_deposit_timestamp"]
        if table_name == "3_funding_account_activity"
        else []
    )

    summary = aggregate_by_client(
        df,
        prefix=prefix,
        exclude_columns=exclusions
    )

    # Enforce one row per client before merging
    assert summary["client_id"].is_unique, (
        f"{table_name} summary has duplicate client IDs"
    )

    merged_df = merged_df.merge(
        summary,
        on="client_id",
        how="left",
        validate="one_to_one"
    )

# Merge the modelling outcomes table
outcomes = tables["9_modelling_outcomes"].copy()

# Exact duplicate rows were already removed; the outcomes table
# should have one record per client.
if outcomes["client_id"].duplicated().any():
    raise ValueError(
        "The outcomes table has multiple records per client. "
        "Investigate these before merging."
    )

merged_df = merged_df.merge(
    outcomes,
    on="client_id",
    how="left",
    validate="one_to_one",
    suffixes=("", "_outcome")
)

print("Merge completed successfully.")
print("Final dataset shape:", merged_df.shape)
print("Unique clients:", merged_df["client_id"].nunique())
print("Duplicate client IDs:", merged_df["client_id"].duplicated().sum())

display(merged_df.head())


Merge completed successfully.
Final dataset shape: (2104, 120)
Unique clients: 2104
Duplicate client IDs: 0


,client_id,registration_date,kyc_completion_date,kyc_status,acquisition_channel,region,age,stated_goal,risk_appetite,app_event_timestamp_min,...,compliance_alert_type_mode,compliance_alert_disposition_nunique,compliance_alert_disposition_mode,compliance_case_outcome_nunique,compliance_case_outcome_mode,activation_outcome,first_deposit_timestamp,dormancy_outcome,dormancy_timestamp,observation_end_date
0,CL_004000,2025-09-03 22:22:33,2025-09-04 00:36:12,Completed,Paid social,FS,27.0,Retirement,Not stated,2025-09-07 19:24:31,...,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,2026-08-31 23:59:59
1,CL_004001,2024-04-12 02:09:20,NaN,In Progress,Paid search,GP,34.0,Long-term investing,Moderate,2024-04-19 03:22:35,...,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,2026-08-31 23:59:59
2,CL_004002,2023-10-15 16:24:14,2023-10-16 02:19:23,Completed,Paid search,LP,25.0,Unknown,Moderate,2023-10-16 14:59:26,...,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,2026-08-31 23:59:59
3,CL_004003,2024-11-28 04:27:10,2024-11-28 19:03:06,Completed,Influencer,WC,23.0,Retirement,Aggressive,2024-12-01 04:53:18,...,NaN,NaN,NaN,NaN,NaN,True,2024-11-30 04:52:09,False,NaN,2026-08-31 23:59:59
4,CL_004004,2024-04-16 22:31:40,2024-04-16 23:48:27,Completed,Referral,KZN,22.0,Education fund,Aggressive,2024-05-01 13:06:52,...,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,2026-08-31 23:59:59


### Validate the merged dataset

These checks confirm that each final row has a unique, non-missing client ID. Outcome distributions help establish which labels are available, while the missingness summary highlights columns that may need investigation. The `info()` output provides data types and non-null counts. These checks are structural and should be followed by business-rule validation and a target-leakage review.

In [8]:
# Check key integrity
assert merged_df["client_id"].is_unique
assert merged_df["client_id"].notna().all()

# Check target availability
print("Activation outcomes:")
print(merged_df["activation_outcome"].value_counts(dropna=False))

print("\nDormancy outcomes:")
print(merged_df["dormancy_outcome"].value_counts(dropna=False))

# Review missing values
missing_summary = (
    merged_df.isna()
    .sum()
    .to_frame("missing_count")
)

missing_summary["missing_percentage"] = (
    missing_summary["missing_count"] / len(merged_df) * 100
).round(2)

display(
    missing_summary
    .sort_values("missing_percentage", ascending=False)
    .head(20)
)

# Review dataset structure
merged_df.info()


Activation outcomes:
activation_outcome
False    1288
True      816
Name: count, dtype: int64

Dormancy outcomes:
dormancy_outcome
NaN      1288
False     673
True      143
Name: count, dtype: int64


,missing_count,missing_percentage
compliance_alert_disposition_mode,1996,94.87
compliance_case_outcome_nunique,1996,94.87
compliance_case_outcome_mode,1996,94.87
compliance_alert_disposition_nunique,1996,94.87
compliance_alert_timestamp_max,1996,94.87
compliance_alert_type_nunique,1996,94.87
compliance_alert_type_mode,1996,94.87
compliance_alert_timestamp_min,1996,94.87
dormancy_timestamp,1961,93.20
subscription_end_timestamp_max,1933,91.87


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2104 entries, 0 to 2103
Columns: 120 entries, client_id to observation_end_date
dtypes: bool(1), datetime64[ns](16), float64(53), int64(14), object(35), string(1)
memory usage: 1.9+ MB


In [9]:
output_path = Path("STADIOEquities_Merged_Client_Data.csv")

merged_df.to_csv(output_path, index=False)

print(f"Saved merged dataset to: {output_path.resolve()}")
print(f"Rows: {len(merged_df):,}")
print(f"Columns: {len(merged_df.columns):,}")


Saved merged dataset to: /content/STADIOEquities_Merged_Client_Data.csv
Rows: 2,104
Columns: 120
